Harware requirements
1. The primary disk storage needs to be greater than 256GB+
2. Should be able to expose external ports

Currently only Azure, GCP allow these requiremts on brev 

In [ ]:
#Check the NVIDIA driver version(Must be 535)
!nvidia-smi

In [1]:
#Incase you need to uninstall and reinstall 
!sudo apt-get purge nvidia-*
#Then check to make sure everything has been removed 
!lsmod | grep nvidia

Reading package lists... Done
Building dependency tree       
Reading state information... Done
E: Unable to locate package nvidia-blueprint-vss-2.1.0.tgz
E: Couldn't find any package by glob 'nvidia-blueprint-vss-2.1.0.tgz'
E: Couldn't find any package by regex 'nvidia-blueprint-vss-2.1.0.tgz'
nvidia_uvm           1540096  126
nvidia_drm             77824  24
nvidia_modeset       1306624  3 nvidia_drm
nvidia              56721408  7182 nvidia_uvm,nvidia_modeset
drm_kms_helper        307200  1 nvidia_drm
drm                   618496  28 drm_kms_helper,nvidia,nvidia_drm


In [ ]:
#Install again 
!wget https://in.download.nvidia.com/tesla/535.161.08/NVIDIA-Linux-x86_64-535.161.08.run

#Run the following commands:

!chmod 755 NVIDIA-Linux-x86_64-535.161.08.run
!sudo ./NVIDIA-Linux-x86_64-535.161.08.run --no-cc-version-check

In [ ]:
#Export your NGC key that has access EA VSS
export NGC_API_KEY=<key>

In [ ]:
# Install microk8s
sudo snap install microk8s --classic
# Enable nvidia and hostpath-storage add-ons
sudo microk8s enable nvidia
sudo microk8s enable hostpath-storage
# Install kubectl
sudo snap install kubectl --classic
# Verify microk8s is installed correctly
sudo microk8s kubectl get pod -A

IMPORTANT- Ensure all pods are either Running or in Completed stage

In [ ]:
# Create the NGC image pull secret

sudo microk8s kubectl create secret docker-registry ngc-docker-reg-secret --docker-server=nvcr.io --docker-username='$oauthtoken' --docker-password=$NGC_API_KEY

# Create the neo4j db credentials secret

sudo microk8s kubectl create secret generic graph-db-creds-secret --from-literal=username=neo4j --from-literal=password=password

# Create NGC Secret

sudo microk8s kubectl create secret generic ngc-api-key-secret --from-literal=NGC_API_KEY=$NGC_API_KEY

In [ ]:
# Fetch the VSS Blueprint Helm Chart

sudo microk8s helm fetch https://helm.ngc.nvidia.com/nvidia/blueprint/charts/nvidia-blueprint-vss-2.1.0.tgz --username='$oauthtoken' --password=$NGC_API_KEY

# Install the Helm Chart

sudo microk8s helm install vss-blueprint nvidia-blueprint-vss-2.1.0.tgz --set global.ngcImagePullSecretName=ngc-docker-reg-secret

In [ ]:
In the access tab expose port 31630 to access the web UI 